# 1장. LLM, 다음 글자를 맞히는 기계

LLM이 무엇인지, 어떻게 여기까지 왔는지, 이 책에서 무엇을 만들지 살펴봅니다. 마지막에는 작은 GPT를 2분 동안 학습시켜 **학습 전과 후의 글**을 직접 비교합니다.
코드는 이해하려 하지 말고 실행하고 관찰하세요. 2장부터 하나씩 다시 만들며 이해합니다.

**GPU가 없어도 됩니다.** T4 GPU를 연결하면 같은 2분 동안 더 많이 학습해서 글이 더 그럴듯해집니다.

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0) · 구조 참고: [nanoGPT](https://github.com/karpathy/nanoGPT)


## 1.1 LLM은 다음 글자를 맞히는 기계다

휴대폰 키보드에 "오늘 저녁에"까지 쓰면 "뭐", "먹을까" 같은 다음 낱말을 추천해 줍니다. ChatGPT 같은 **LLM**(Large Language Model, 거대 언어 모델)이 하는 일도 같습니다. 지금까지의 글을 보고 **다음에 올 글자(토큰)를 맞히고**, 맞힌 글자를 붙여 또 다음 글자를 맞힙니다. 이것을 수없이 반복해 긴 글을 만듭니다.


## 1.2 LLM은 어떻게 여기까지 왔나

| 때 | 일어난 일 |
|---|---|
| 1966 | ELIZA: 사람이 정한 규칙으로 대답하는 대화 프로그램 |
| 1990~2000년대 | 통계 언어 모델: 앞의 낱말 몇 개를 세어 다음 낱말의 확률을 계산 |
| 2010년대 | 신경망 언어 모델(RNN, LSTM): 글을 차례로 읽으며 기억 |
| 2017 | Transformer: 어텐션으로 앞의 모든 글자를 한꺼번에 돌아봄 (이 책 4~8장) |
| 2018~2020 | GPT-1·2·3: 많은 글로 사전학습한 Transformer를 점점 크게 (이 책 9~10장) |
| 2022 | ChatGPT: 요청을 따르도록 파인튜닝해 대화가 가능해짐 (이 책 11~12장) |
| 2023 이후 | 여러 회사의 LLM과 공개 모델, 그림·소리를 함께 다루는 모델, 오래 생각하는 추론 모델 |


## 1.3 이 책에서 할 일

한국어 어린이 이야기를 쓰는 **작은 GPT를 처음부터 만들고**, 요청을 알아듣는 **이야기 챗봇**까지 완성합니다.

| 장 | 만드는 것 |
|---|---|
| 2장 | 이야기 데이터와 글자 사전 |
| 3장 | 다음 글자 맞히기 문제, 임베딩 |
| 4~7장 | 어텐션 → 멀티헤드 → 블록 → GPT |
| 8장 | 같은 GPT를 빠르게 계산하기 |
| 9~10장 | 사전학습과 글 생성 |
| 11~12장 | 요청을 따르는 이야기 챗봇 |

부품은 여러분이 장마다 직접 코드로 만들어요. 같은 코드를 `modu_llm` 패키지에도 담아 두었으니, 다음 장에서는 그 부품을 불러다 씁니다.


## 1.5 미리 맛보기: 2분 만에 작은 GPT 학습시키기

책의 부품으로 작은 GPT를 만들어 2분 동안 학습시켜요. 각 셀의 주석에 자세히 배우는 장을 적었습니다.

**핵심 코드**: `train(model, optimizer, train_data, val_data, minutes=2, ...)`


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from modu_llm import load_stories, Vocab, encode_texts, split_data
from modu_llm import GPT, GPTConfig, count_params, generate, next_char_probs, train
from modu_llm import get_device, plot_history

device = get_device()


In [ ]:
stories = load_stories(3000)               # 어린이 이야기 3,000편 (2장)
vocab = Vocab.build()                      # 글자 사전 (2장)
data = encode_texts(stories, vocab)        # 글자를 번호로 (2장)
train_data, val_data = split_data(data)    # 학습용 90%, 검증용 10% (3장)
print(f"이야기 {len(stories):,}편 | 글자 {len(data):,}개")
print(stories[0][:100])


In [ ]:
cfg = GPTConfig(block_size=64, n_embd=128, n_head=4, n_layer=4)   # 작은 GPT의 크기 (7장)
model = GPT(cfg).to(device)                                       # 작은 GPT (4~8장)
print(f"파라미터 수: {count_params(model):,}")

before = generate(model, vocab, "어느 날, ", max_new_tokens=100)    # 글 만들기 (10장)
print("[학습 전]", before)


모델은 "어느 날, 작은 토끼" 다음에 어떤 글자가 올 거라고 생각할까요? 학습 전의 모델에게 물어봐요.


In [ ]:
for ch, p in next_char_probs(model, vocab, "어느 날, 작은 토끼"):
    percent = p * 100
    print(repr(ch), f"{percent:.1f}%")


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
step, history = train(model, optimizer, train_data, val_data,    # 학습 (9장)
                      minutes=2, ckpt_path="tiny.pt", eval_every=200)


## 1.6 학습 전과 후, 무엇이 달라졌나

손실(틀린 정도)이 줄어드는 그래프를 보고, 학습 전과 같은 질문을 다시 해 봐요. 띄어쓰기, 받침, '-어요' 같은 어미가 보이나요?


In [ ]:
plot_history(history)                      # 손실 그래프 (9장)

after = generate(model, vocab, "어느 날, ", max_new_tokens=100)
print("[학습 후]", after)


In [ ]:
for ch, p in next_char_probs(model, vocab, "어느 날, 작은 토끼"):
    percent = p * 100
    print(repr(ch), f"{percent:.1f}%")


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

- 시작 문장을 바꿔 보세요. 예: `generate(model, vocab, "엄마가 ")`, `generate(model, vocab, "밤하늘에 ")`
- `temperature=0.3`이나 `temperature=1.5`를 넣어 보세요. 글이 어떻게 달라지나요?
- `next_char_probs`에 다른 시작 문장을 넣어 다음 글자 후보를 살펴보세요.


In [ ]:
story = generate(model, vocab, "밤하늘에 ", max_new_tokens=100, temperature=0.3)
print(story)


## 실제 LLM에서는

- 실제 LLM은 글자보다 큰 단위(하위 단어)를 토큰으로 쓰고, 파라미터가 수십억 개 이상이며, 수조 개의 토큰으로 몇 주에서 몇 달 동안 학습해요.
- 그래도 핵심 원리, **다음 토큰을 맞히도록 학습하고 하나씩 이어 붙여 글을 만든다**는 점은 이 장의 작은 GPT와 같습니다.
